## WaDi A2 - Pipeline Notebook 1: Ingest & Stage

**Note: This dataset is superseding the previous pipeline using WaDi A1**  
This dataset serves the same purpose but cleans up some faulty sensors that plagued the original dataset.  

The notebook will cover the following:  
* **Dataset:** WaDi.A2_19 Nov 2019 - Water Distribution testbed, iTrust Labs, SUTD
* **Scope:** Stages 0-2. Ingests raw CSV files, cleans and types the data, drops uninformative sensor columns, and assigns a temporal train/test split following the standard WaDi evaluation protocol.
* **Split strategy:** Normal operation rows (label=0) -> train. Attack period rows (label=1) -> test. This matches the convention used in the ICS anomaly detection literature
* **Output:** A staged Parquet file with a `split` column, ready for fault injection.

# Stage 0 - Setup

## 0.1 - Imports and Paths

In [15]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

# ── Paths ─────────────────────────────────────────────────────────────────────
WORK_DIR    = Path("work")
PROJECT_DIR = WORK_DIR / "wadi_A2"
DATA_DIR    = PROJECT_DIR / "data"
RAW_DIR     = DATA_DIR / "raw"
STAGED_DIR  = DATA_DIR / "staged"
REF_DIR     = DATA_DIR / "reference"
RUN_DIR     = REF_DIR / "pipeline_runs"

for p in [RAW_DIR, STAGED_DIR, REF_DIR, RUN_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Raw:    ", RAW_DIR)
print("Staged: ", STAGED_DIR)
print("Ref:    ", REF_DIR)

Project: work/wadi_A2
Raw:     work/wadi_A2/data/raw
Staged:  work/wadi_A2/data/staged
Ref:     work/wadi_A2/data/reference


## 0.2 Helper Utilities  
Reusable functions used throughout the pipeline

In [16]:
class PipelineError(RuntimeError):
    pass

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def sha16(x: str) -> str:
    return hashlib.sha256(x.encode("utf-8")).hexdigest()[:16]

def write_json(path: Path, obj: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=str))

def read_json(path: Path) -> dict:
    return json.loads(path.read_text())

def require_columns(df: pd.DataFrame, cols: list[str], context: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise PipelineError(f"[{context}] Missing required columns: {missing}")

print("Helpers ready.")

Helpers ready.


## 0.3 - Configuration   
Pipeline constants. Dataset: WaDi.A2_19 Nov 2019 - Water Distribution testbed, iTrust Labs, SUTD

In [17]:
# Dataset Identity
DATASET_NAME   = "WaDi.A2_19 Nov 2019"
DATA_SOURCE    = "iTrust Labs, SUTD"
DATASET_URL    = "https://itrust.sutd.edu.sg/itrust-labs_datasets/dataset_info/"
DOWNLOAD_DATE  = "2026-02-28"

# Expected raw files
EXPECTED_FILES = [
    "WADI_14days_new.csv",
    "WADI_attackdataLABLE.csv",
    "table_WADI.pdf",
]

# Run ID
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")

print(f"Dataset:  {DATASET_NAME}")
print(f"Source:   {DATA_SOURCE}")
print(f"Run ID:   {RUN_ID}")
print(f"Split:    normal operation → train | attack period → test")

Dataset:  WaDi.A2_19 Nov 2019
Source:   iTrust Labs, SUTD
Run ID:   20260303_165154_utc
Split:    normal operation → train | attack period → test


# Stage 1 - Ingest: Acquire Raw Data  
Verifies all expected raw files are present and documents their provenance.  
The raw files are not modified - this stage is read-only

## 1.1 - Verify Raw Files

In [18]:
# Verify all expected raw files are present
missing = [f for f in EXPECTED_FILES if not (RAW_DIR / f).exists()]

if missing:
    raise PipelineError(f"Missing required files in {RAW_DIR}: {missing}")

print("All required files present:\n")
for fname in EXPECTED_FILES:
    fpath = RAW_DIR / fname
    size_mb = fpath.stat().st_size / 1e6
    print(f"  {fname:<35s}  {size_mb:>8.1f} MB")

All required files present:

  WADI_14days_new.csv                     493.7 MB
  WADI_attackdataLABLE.csv                109.3 MB
  table_WADI.pdf                            0.0 MB


## 1.2 - Write Raw Metadata  
Documents the source, download date, and file fingerprints for reproducibility

In [19]:
file_meta = {}
for fname in EXPECTED_FILES:
    fpath = RAW_DIR / fname
    file_meta[fname] = {
        "size_bytes": fpath.stat().st_size,
        "sha16":      sha16(fpath.read_bytes().hex()),
        "path":       str(fpath),
    }

metadata_raw = {
    "run_id":        RUN_ID,
    "created_at_utc": utc_now_iso(),
    "dataset_name":  DATASET_NAME,
    "source":        DATA_SOURCE,
    "source_url":    DATASET_URL,
    "download_date": DOWNLOAD_DATE,
    "files":         file_meta,
    "notes": [
        "Dataset requires registration with iTrust Labs.",
        "Files manually downloaded and placed in RAW_DIR.",
        "Normal operations: WADI_14days_new.csv — unstable periods removed vs A1.",
        "Attack file: WADI_attackdataLABLE.csv — embedded label column (Attack LABLE).",
        "Attack label encoding: -1 = cyber attack, 1 = normal operation within attack window.",
    ],
}

meta_path = RAW_DIR / f"metadata_raw_{RUN_ID}.json"
write_json(meta_path, metadata_raw)
print(f"Metadata written: {meta_path}")

Metadata written: work/wadi_A2/data/raw/metadata_raw_20260303_165154_utc.json


# Stage 2 - Stage: Parse and Normalize  
* Loads raw CSVs
* cleans column names
* parses timestamps
* combines normal and attack data into a single DataFrame
* drops uninformative sensor columns
* defines the SENSOR_COLS list
* adds time features
* assigns the stratified train/val/test split
* writes the staged parquet

## 2.1 - Define Staging Function  
Cleans the raw WaDi CSV structure:  
* Strips the windows path prefix from column names
* Parses separate Date + Time columns into a single UTC timestamp
* Drops the row-number column
* Casts all sensor columns to float32

In [20]:
# Windows path prefix present on all sensor column names
WADI_COL_PREFIX = "\\\\WIN-25J4RO10SBF\\LOG_DATA\\SUTD_WADI\\LOG_DATA\\"

def stage_wadi_data(df_raw: pd.DataFrame, dataset_id: str) -> pd.DataFrame:
    """
    Parse and normalize one raw WaDi CSV (normal or attack).
    Returns a clean, typed DataFrame with UTC timestamps.
    """
    df = df_raw.copy()

    # Strip Windows path prefix from column names 
    df.columns = [
        c.replace(WADI_COL_PREFIX, "").strip()
        if c.startswith("\\\\") else c.strip()
        for c in df.columns
    ]

    # Drop row-number column (unnamed or 'Row') 
    drop_candidates = [c for c in df.columns if c.strip() in ("", "Row") or
                       c.startswith("Unnamed")]
    df = df.drop(columns=drop_candidates, errors="ignore")

    # Parse timestamp 
    date_parsed = pd.to_datetime(df["Date"], format="mixed", dayfirst=False)
    time_parsed = pd.to_timedelta("00:" + df["Time"].fillna("00:00.0").astype(str))
    df["timestamp"] = (date_parsed + time_parsed).dt.tz_localize("UTC")
    
    df = df.drop(columns=["Date", "Time"], errors="ignore")
    
    # Add source identifier 
    df["dataset_id"] = dataset_id

    # Sort by timestamp 
    df = df.sort_values("timestamp").reset_index(drop=True)

    # Cast sensor columns to float32 
    meta_cols = {"timestamp", "dataset_id", "Attack LABLE (1:No Attack, -1:Attack)"}
    sensor_cols = [c for c in df.columns if c not in meta_cols]
    for col in sensor_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

    return df

print("Staging function defined.")

Staging function defined.


## 2.2 - Load and Stage Raw Data  
Loads both CSV files and applies the staging function. The normal operations file has metadata rows at the top that must be skipped

In [21]:
# Peek at the first 10 raw lines to understand the file structure
with open(RAW_DIR / "WADI_14days_new.csv", "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line[:120]!r}")
        if i >= 9:
            break

Line 0: 'Row,Date,Time,1_AIT_001_PV,1_AIT_002_PV,1_AIT_003_PV,1_AIT_004_PV,1_AIT_005_PV,1_FIT_001_PV,1_LS_001_AL,1_LS_002_AL,1_LT'
Line 1: '1,9/25/2017,00:00.0,171.155,0.619473,11.5759,504.645,0.318319,0.00115685,0,0,47.8911,1,1,1,1,1,1,1,1,2,1,2464.88,9.08055'
Line 2: '2,9/25/2017,00:01.0,171.155,0.619473,11.5759,504.645,0.318319,0.00115685,0,0,47.8911,1,1,1,1,1,1,1,1,2,1,2464.88,9.08055'
Line 3: '3,9/25/2017,00:02.0,171.155,0.619473,11.5759,504.645,0.318319,0.00115685,0,0,47.8911,1,1,1,1,1,1,1,1,2,1,2464.88,9.08055'
Line 4: '4,9/25/2017,00:03.0,171.155,0.607477,11.5725,504.673,0.318438,0.00120685,0,0,47.7503,1,1,1,1,1,1,1,1,2,1,2477.67,8.61453'
Line 5: '5,9/25/2017,00:04.0,171.155,0.607477,11.5725,504.673,0.318438,0.00120685,0,0,47.7503,1,1,1,1,1,1,1,1,2,1,2477.67,8.61453'
Line 6: '6,9/25/2017,00:05.0,171.155,0.607477,11.5725,504.673,0.318438,0.00120685,0,0,47.7503,1,1,1,1,1,1,1,1,2,1,2477.67,8.61453'
Line 7: '7,9/25/2017,00:06.0,171.155,0.607477,11.5725,504.673,0.318438,0.00120685,0

In [22]:
print("\n--- Attack file preview ---")
with open(RAW_DIR / "WADI_attackdataLABLE.csv", "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {line[:150]!r}")
        if i >= 4:
            break


--- Attack file preview ---
Line 0: '0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,5'
Line 1: 'Row ,Date ,Time,1_AIT_001_PV,1_AIT_002_PV,1_AIT_003_PV,1_AIT_004_PV,1_AIT_005_PV,1_FIT_001_PV,1_LS_001_AL,1_LS_002_AL,1_LT_001_PV,1_MV_001_STATUS,1_MV'
Line 2: '1,10/9/17,00:00.0,164.21,0.529486,11.9972,482.48,0.331167,0.00127323,0,0,48.482,1,1,1,1,1,1,1,1,1,1,2538.7,35.6659,0.240392,0.2,68.5648,0.306246,0.1,1'
Line 3: '2,10/9/17,00:01.0,164.21,0.529486,11.9972,482.48,0.331167,0.00127323,0,0,48.482,1,1,1,1,1,1,1,1,1,1,2538.7,35.6659,0.240392,0.2,68.5648,0.306246,0.1,1'
Line 4: '3,10/9/17,00:02.0,164.21,0.529486,11.9972,482.48,0.331167,0.00127323,0,0,48.482,1,1,1,1,1,1,1,1,1,1,2538.7,35.6659,0.240392,0.2,68.5648,0.306246,0.1,1'


In [23]:
# 1. Load normal operations CSV
print("Loading normal operations data...")
df_raw_normal = pd.read_csv(RAW_DIR / "WADI_14days_new.csv", low_memory=False)
print(f"  Raw shape: {df_raw_normal.shape}")

# 2. Load attack data CSV
print("\nLoading attack data...")
df_raw_attack = pd.read_csv(RAW_DIR / "WADI_attackdataLABLE.csv", skiprows=0, header=1, low_memory=False)
print(f"  Raw shape: {df_raw_attack.shape}")
print(f"\n  'Attack LABLE' value counts:")
print(df_raw_attack["Attack LABLE (1:No Attack, -1:Attack)"].value_counts().sort_index())

# 3. Stage both
print("\nStaging normal data...")
df_normal = stage_wadi_data(df_raw_normal, dataset_id="normal")
print(f"  Staged shape: {df_normal.shape}")
print(f"  Time range: {df_normal['timestamp'].min()} → {df_normal['timestamp'].max()}")

print("\nStaging attack data...")
df_attack = stage_wadi_data(df_raw_attack, dataset_id="attack")
print(f"  Staged shape: {df_attack.shape}")
print(f"  Time range: {df_attack['timestamp'].min()} → {df_attack['timestamp'].max()}")

Loading normal operations data...
  Raw shape: (784571, 130)

Loading attack data...
  Raw shape: (172803, 131)

  'Attack LABLE' value counts:
Attack LABLE (1:No Attack, -1:Attack)
-1      9977
 1    162826
Name: count, dtype: int64

Staging normal data...
  Staged shape: (784571, 129)
  Time range: 2017-09-25 00:00:00+00:00 → 2017-10-07 00:59:59+00:00

Staging attack data...
  Staged shape: (172803, 130)
  Time range: 2017-10-09 00:00:00+00:00 → 2017-10-11 00:59:59+00:00


## 2.3 - Combine and Assign Labels  
* Concatenates normal and attack DataFrames into a single dataset
* Labels are derived from the embedded `Attack LABLE (1:No Attack, -1:Attack)` column in the attack file:
  * Normal file rows &rarr; `label=0`
  * Attack file rows where `Attack LABLE = -1` &rarr; `label=1` (cyber attack)
  * Attack file rows where `Attack LABLE = 1` &rarr; `label=0` (normal within attack window)
* Drops `Attack LABLE` and `dataset_id` columns, both encode the label directly and would be data leakage if carried into the feature matrix

In [24]:
# 1. Assign label to normal file
df_normal = df_normal.copy()
df_normal["label"] = 0

# 2. Assign labels to attack file using embedded label column
#    -1 = cyber attack -> label 1
#     1 = normal within attack window -> label 0
df_attack = df_attack.copy()
df_attack["label"] = df_attack["Attack LABLE (1:No Attack, -1:Attack)"].apply(
    lambda x: 1 if x == -1.0 else 0
).astype("int8")
df_attack = df_attack.drop(columns=["Attack LABLE (1:No Attack, -1:Attack)"])

# 3. Combine 
df = pd.concat([df_normal, df_attack], ignore_index=True)
df = df.sort_values("timestamp").reset_index(drop=True)

# 4. Drop dataset_id — encodes source file, direct leakage
df = df.drop(columns=["dataset_id"])

# 5. Cast label to int8
df["label"] = df["label"].astype("int8")

print(f"Combined shape: {df.shape}")
print(f"Time range:     {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"\nLabel counts:")
print(f"  Normal (0): {(df['label']==0).sum():>9,}")
print(f"  Attack (1): {(df['label']==1).sum():>9,}")

Combined shape: (957374, 129)
Time range:     2017-09-25 00:00:00+00:00 → 2017-10-11 00:59:59+00:00

Label counts:
  Normal (0):   947,397
  Attack (1):     9,977


## 2.4 - Drop Uninformative Sensor Columns  
Identifies and removes sensor columns that carry no predictive signal:  
* **100% NaN** - sensor was never recorded or permanently offline
* **Constant value** - no variation across the entire dataset, zero discriminative power

These columns are documented explicitly before removal so the decision is reproducible and citable. They will be excluded from SENSOR_COLS permanently.  

In [25]:
# Identify all sensor columns (everything except timestamp and label)
meta_cols = {"timestamp", "label"}
all_sensor_candidates = [c for c in df.columns if c not in meta_cols]

# Find 100% NaN columns 
null_counts = df[all_sensor_candidates].isnull().sum()
fully_null = null_counts[null_counts == len(df)].index.tolist()

# Find constant columns (zero variance)
constant = [
    c for c in all_sensor_candidates
    if c not in fully_null and df[c].nunique(dropna=True) <= 1
]

print(f"100% NaN columns ({len(fully_null)}):")
for c in fully_null:
    print(f"  {c}")

print(f"\nConstant-value columns ({len(constant)}):")
for c in constant:
    val = df[c].dropna().unique()
    print(f"  {c:<35s}  value={val[0] if len(val) else 'NaN'}")

print(f"\nTotal to drop: {len(fully_null) + len(constant)}")

100% NaN columns (4):
  2_LS_001_AL
  2_LS_002_AL
  2_P_001_STATUS
  2_P_002_STATUS

Constant-value columns (25):
  1_LS_001_AL                          value=0.0
  1_LS_002_AL                          value=0.0
  1_P_002_STATUS                       value=1.0
  1_P_004_STATUS                       value=1.0
  2_MV_001_STATUS                      value=1.0
  2_MV_002_STATUS                      value=1.0
  2_MV_004_STATUS                      value=2.0
  2_MV_005_STATUS                      value=2.0
  2_MV_009_STATUS                      value=2.0
  2_P_004_STATUS                       value=1.0
  2_SV_101_STATUS                      value=1.0
  2_SV_201_STATUS                      value=1.0
  2_SV_301_STATUS                      value=1.0
  2_SV_401_STATUS                      value=1.0
  2_SV_501_STATUS                      value=1.0
  2_SV_601_STATUS                      value=1.0
  3_LS_001_AL                          value=1.0
  3_MV_001_STATUS                      value=1.0
  3_

In [26]:
# Drop uninformative columns 
cols_to_drop = fully_null + constant

# Document what we're dropping before removing
drop_log = {
    "fully_null":  fully_null,
    "constant":    constant,
    "total_dropped": len(cols_to_drop),
}

df = df.drop(columns=cols_to_drop)

n_before = len(df.columns) + len(cols_to_drop)
print(f"Columns before drop: {n_before}")
print(f"Columns dropped:     {len(cols_to_drop)}")
print(f"Columns remaining:   {len(df.columns)}")
print(f"\nRemaining columns: timestamp, label + {len(df.columns) - 2} sensor columns")

Columns before drop: 129
Columns dropped:     29
Columns remaining:   100

Remaining columns: timestamp, label + 98 sensor columns


## 2.5 - Define and Freeze SENSOR_COLS  
Defines the canonical list of sensor columns used by all downstream modules.  
Written to a reference JSON so fault injection and curation modules load the same column list without hardcoding it.

In [27]:
# Define canonical sensor column list 
SENSOR_COLS = [c for c in df.columns if c not in {"timestamp", "label"}]

print(f"SENSOR_COLS count: {len(SENSOR_COLS)}")
print(f"\nFirst 10: {SENSOR_COLS[:10]}")
print(f"Last 10:  {SENSOR_COLS[-10:]}")

# Write to reference JSON for downstream notebooks 
sensor_cols_path = REF_DIR / "sensor_cols.json"
write_json(sensor_cols_path, {
    "run_id":       RUN_ID,
    "dataset":      DATASET_NAME,
    "sensor_cols":  SENSOR_COLS,
    "n_sensors":    len(SENSOR_COLS),
    "dropped_fully_null": drop_log["fully_null"],
    "dropped_constant":   drop_log["constant"],
    "notes": [
        "100% NaN columns dropped — sensor never recorded or permanently offline.",
        "Constant-value columns dropped — no variation, zero discriminative power.",
        "This list is the canonical feature set for all downstream notebooks.",
    ]
})

print(f"\nSENSOR_COLS written to: {sensor_cols_path}")

SENSOR_COLS count: 98

First 10: ['1_AIT_001_PV', '1_AIT_002_PV', '1_AIT_003_PV', '1_AIT_004_PV', '1_AIT_005_PV', '1_FIT_001_PV', '1_LT_001_PV', '1_MV_001_STATUS', '1_MV_002_STATUS', '1_MV_003_STATUS']
Last 10:  ['2B_AIT_004_PV', '3_AIT_001_PV', '3_AIT_002_PV', '3_AIT_003_PV', '3_AIT_004_PV', '3_AIT_005_PV', '3_FIT_001_PV', '3_LT_001_PV', 'LEAK_DIFF_PRESSURE', 'TOTAL_CONS_REQUIRED_FLOW']

SENSOR_COLS written to: work/wadi_A2/data/reference/sensor_cols.json


## 2.6 - Add Time Features  
Adds two time-derived columns:  
* `observation_day` - calendar date, used for canary checks and daily row counts
* `seconds_since_start` - ordinal position from dataset start, captures startup vs steady-state behavior without encoding clock time.


In [30]:
# Add time features 
df = df.dropna(subset=["timestamp"]).reset_index(drop=True)
print(f"Rows after dropping NaT timestamps: {len(df):,}")
df["observation_day"] = df["timestamp"].dt.date

t0 = df["timestamp"].min()
df["seconds_since_start"] = (
    (df["timestamp"] - t0).dt.total_seconds().astype("float32")
)

print(f"observation_day range:     {df['observation_day'].min()} → {df['observation_day'].max()}")
print(f"seconds_since_start range: {df['seconds_since_start'].min():.0f} → {df['seconds_since_start'].max():.0f}")
print(f"  ({df['seconds_since_start'].max() / 86400:.1f} days)")

Rows after dropping NaT timestamps: 957,372
observation_day range:     2017-09-25 → 2017-10-11
seconds_since_start range: 0 → 1385999
  (16.0 days)


## 2.7 - Train/Test Split

First few cells to look into the dataset and understand the attack data.

In [31]:
# Identify the 7 attack sequences
attack_rows = df[df["label"] == 1].sort_values("timestamp").reset_index(drop=True)
time_diffs = attack_rows["timestamp"].diff()
gap_indices = time_diffs[time_diffs > pd.Timedelta(minutes=1)].index.tolist()
boundaries = [0] + gap_indices + [len(attack_rows)]

sequences = {}
for i in range(len(boundaries) - 1):
    start_idx = boundaries[i]
    end_idx   = boundaries[i+1] - 1
    sequences[i+1] = {
        "start": attack_rows.loc[start_idx, "timestamp"],
        "end":   attack_rows.loc[end_idx,   "timestamp"],
        "rows":  end_idx - start_idx + 1,
    }

print("Attack sequences:")
for seq_id, seq in sequences.items():
    print(f"  Seq {seq_id}: {seq['start']} → {seq['end']}  ({seq['rows']:,} rows)")

Attack sequences:
  Seq 1: 2017-10-09 00:25:03+00:00 → 2017-10-09 00:50:03+00:00  (1,501 rows)
  Seq 2: 2017-10-10 00:00:00+00:00 → 2017-10-10 00:59:59+00:00  (5,134 rows)
  Seq 3: 2017-10-11 00:00:00+00:00 → 2017-10-11 00:05:03+00:00  (304 rows)
  Seq 4: 2017-10-11 00:07:33+00:00 → 2017-10-11 00:10:55+00:00  (203 rows)
  Seq 5: 2017-10-11 00:16:03+00:00 → 2017-10-11 00:47:03+00:00  (2,690 rows)
  Seq 6: 2017-10-11 00:55:03+00:00 → 2017-10-11 00:56:30+00:00  (88 rows)
  Seq 7: 2017-10-11 00:59:03+00:00 → 2017-10-11 00:59:59+00:00  (57 rows)


## 2.7a - Train/Test Split

**Strategy: Manual sequence-level splitting**

The standard WaDi protocol (normal → train, attack → test) is not used here because it prevents the model from seeing any attack examples during training, which is unrealistic for a supervised classifier.

Instead, the dataset is split at the **attack sequence level**:

* 7 distinct attack sequences were identified by gaps > 1 minute between consecutive attack rows
* Each sequence is assigned **entirely** to train or test — no sequence is split across the boundary
* Sequences were manually assigned to ensure each split contains attacks of varying duration
* Normal rows are assigned based on temporal proximity to their neighboring attack sequences

**Sequence assignments:**

| Seq | Time range | Duration | Rows | Split |
|-----|------------|----------|------|-------|
| 1 | Oct 09 00:25–00:50 | 25 min | 1,501 | Train |
| 2 | Oct 10 00:00–00:59 | 60 min | 5,134 | Test |
| 3 | Oct 11 00:00–00:05 | 5 min | 304 | Test |
| 4 | Oct 11 00:07–00:10 | 3 min | 203 | Test |
| 5 | Oct 11 00:16–00:47 | 31 min | 2,690 | Train |
| 6 | Oct 11 00:55–00:56 | 1.5 min | 88 | Train |
| 7 | Oct 11 00:59 | 1 min | 57 | Train |

* **Train attack rows:** 4,336 (43.5%)
* **Test attack rows:** 5,641 (56.5%)

**Documented limitations:**
* Only 7 attack sequences total — granularity is coarse
* Sequences are highly unequal in size (57 to 5,134 rows)
* This design extends naturally to train/val/test by further splitting train sequences

**Why not chronological splitting?**
A strict chronological split places all attacks in test and all normal data in train, leaving the model with no attack examples during training and no normal examples during evaluation — both undermine supervised classification. (Oct 9–11) and all normal data in train (Sept 25 – Oct 7), leaving the model with no attack examples during training and no normal examples during evaluation — both of which undermine supervised classification.

In [32]:
# Manual sequence assignment
TRAIN_SEQUENCES = {1, 5, 6, 7}
TEST_SEQUENCES  = {2, 3, 4}

# Initialize split column
df["split"] = None

# Assign attack rows by sequence
for seq_id, seq in sequences.items():
    mask = (
        (df["label"] == 1) &
        (df["timestamp"] >= seq["start"]) &
        (df["timestamp"] <= seq["end"])
    )
    split_label = "train" if seq_id in TRAIN_SEQUENCES else "test"
    df.loc[mask, "split"] = split_label

# Assign normal rows by temporal proximity to attack sequences
# Normal rows before the first attack sequence → train
# Normal rows within the attack window period → assigned to nearest sequence's split
# Normal rows after the last attack sequence → train (bulk of Sept 25 - Oct 8 data)

first_attack = min(seq["start"] for seq in sequences.values())
last_attack  = max(seq["end"]   for seq in sequences.values())

# Normal rows outside attack window → train
df.loc[
    (df["label"] == 0) & (df["timestamp"] < first_attack),
    "split"
] = "train"

df.loc[
    (df["label"] == 0) & (df["timestamp"] > last_attack),
    "split"
] = "train"

# Normal rows within attack window → assign to split of nearest attack sequence
for seq_id, seq in sequences.items():
    split_label = "train" if seq_id in TRAIN_SEQUENCES else "test"
    mask = (
        (df["label"] == 0) &
        (df["split"].isna()) &
        (df["timestamp"] >= seq["start"]) &
        (df["timestamp"] <= seq["end"])
    )
    df.loc[mask, "split"] = split_label

# Any remaining unassigned normal rows → train
df.loc[(df["label"] == 0) & (df["split"].isna()), "split"] = "train"

print("Split distribution:")
for split in ["train", "test"]:
    n = (df["split"] == split).sum()
    print(f"  {split}: {n:,}")

print("\nAttack rows by split:")
for split in ["train", "test"]:
    n = ((df["split"] == split) & (df["label"] == 1)).sum()
    print(f"  {split}: {n:,}")

Split distribution:
  train: 861,845
  test: 95,527

Attack rows by split:
  train: 4,336
  test: 5,641


## 2.8 - Validate Split  
Confirms the split assignment is correct before saving. Checks:
* All rows have a split assignment
* Both splits exist
* Both normal and attack rows appear in both splits (by design)
* No attack sequence spans both splits

In [33]:
errors = []

# Check all rows assigned
n_unassigned = df["split"].isna().sum()
if n_unassigned > 0:
    errors.append(f"  {n_unassigned} rows have no split assignment")

# Check expected splits exist
for expected_split in ["train", "test"]:
    if expected_split not in df["split"].values:
        errors.append(f"  Split '{expected_split}' is missing entirely")

# Check both labels appear in both splits
for split in ["train", "test"]:
    for label_val, label_name in [(0, "normal"), (1, "attack")]:
        n = ((df["split"] == split) & (df["label"] == label_val)).sum()
        if n == 0:
            errors.append(f"  No {label_name} rows in {split} split")

# Check attack sequence integrity — no sequence spans both splits
for seq_id, seq in sequences.items():
    mask = (
        (df["label"] == 1) &
        (df["timestamp"] >= seq["start"]) &
        (df["timestamp"] <= seq["end"])
    )
    splits_in_seq = df.loc[mask, "split"].unique()
    if len(splits_in_seq) > 1:
        errors.append(f"  Seq {seq_id} spans both splits: {splits_in_seq}")

# Report
if errors:
    print("VALIDATION FAILED:")
    for e in errors:
        print(e)
else:
    print("Split validation PASSED — all checks clean.")

print(f"\nSplit summary:")
for split in ["train", "test"]:
    n_total  = (df["split"] == split).sum()
    n_normal = ((df["split"] == split) & (df["label"] == 0)).sum()
    n_attack = ((df["split"] == split) & (df["label"] == 1)).sum()
    print(f"  {split}: {n_total:>9,} total  |  normal: {n_normal:>9,}  |  attack: {n_attack:>5,}")

Split validation PASSED — all checks clean.

Split summary:
  train:   861,845 total  |  normal:   857,509  |  attack: 4,336
  test:    95,527 total  |  normal:    89,886  |  attack: 5,641


## 2.9 - Write Staged Parquet and Run Log  
Writes the staged dataset to disk and documents the pipeline run.  
The staged parquet is the input to the fault injection module.

In [34]:
# Final column ordering 
ordered_cols = ["timestamp", "observation_day", "seconds_since_start",
                "split", "label"] + SENSOR_COLS
df_staged = df[ordered_cols].copy()

# Write staged parquet 
staged_path = STAGED_DIR / f"wadi_staged_{RUN_ID}.parquet"
df_staged.to_parquet(staged_path, index=False)

size_mb = staged_path.stat().st_size / 1e6
print(f"Staged parquet written: {staged_path}")
print(f"Size:                   {size_mb:.1f} MB")
print(f"Shape:                  {df_staged.shape}")

# Write run log 
run_log = {
    "run_id":           RUN_ID,
    "created_at_utc":   utc_now_iso(),
    "stage":            "Notebook 1 — Ingest and Stage",
    "dataset":          DATASET_NAME,
    "source":           DATA_SOURCE,
    "download_date":    DOWNLOAD_DATE,
    "inputs": {
        "normal_csv":   str(RAW_DIR / "WADI_14days_new.csv"),
        "attack_csv":   str(RAW_DIR / "WADI_attackdataLABLE.csv"),
    },
    "outputs": {
        "staged_parquet":  str(staged_path),
        "sensor_cols_ref": str(sensor_cols_path),
    },
    "dataset_summary": {
        "total_rows":    len(df_staged),
        "total_cols":    len(df_staged.columns),
        "n_sensor_cols": len(SENSOR_COLS),
        "label_counts":  df_staged["label"].value_counts().sort_index().to_dict(),
        "split_counts":  df_staged["split"].value_counts().sort_index().to_dict(),
        "time_start":    str(df_staged["timestamp"].min()),
        "time_end":      str(df_staged["timestamp"].max()),
    },
    "columns_dropped": {
        "fully_null":  drop_log["fully_null"],
        "constant":    drop_log["constant"],
        "total":       drop_log["total_dropped"],
    },
    "split_strategy": "sequence-level manual split — attack sequences assigned to train/test, normal rows by temporal proximity",
    "notes": [
        "dataset_id dropped — directly encodes label, would be leakage.",
        "Attack LABLE column dropped — directly encodes label, would be leakage.",
        "Split is sequence-level: 7 attack sequences manually assigned to train (1,5,6,7) or test (2,3,4).",
        "Train attack rows: 4,336 (43.5%). Test attack rows: 5,641 (56.5%).",
        "Normal rows assigned by temporal proximity to neighboring attack sequences.",
        "Both splits contain normal and attack rows by design — required for supervised classification.",
        "Fault injection notebook reads this parquet and injects label=2 rows into normal rows in both splits.",
    ],
}

run_log_path = RUN_DIR / f"run_{RUN_ID}.json"
write_json(run_log_path, run_log)
print(f"\nRun log written: {run_log_path}")

Staged parquet written: work/wadi_A2/data/staged/wadi_staged_20260303_165154_utc.parquet
Size:                   109.9 MB
Shape:                  (957372, 103)

Run log written: work/wadi_A2/data/reference/pipeline_runs/run_20260303_165154_utc.json


# Stage 3 - Pipeline Reflection
Documents key decisions, assumptions, and risks from this notebook.

In [35]:
reflection = [
    ("Row definition",
     "Each row represents one second of sensor readings from the WaDi A2 water "
     "distribution testbed. Label 0=normal operation, 1=cyber attack (derived "
     "from embedded Attack LABLE column in A2 attack file). "
     "Label 2=injected sensor fault will be added by the fault injection notebook."),

    ("A2 vs A1 differences",
     "Normal file changed to WADI_14days_new.csv — unstable periods removed, "
     "reducing rows from ~1.2M to ~784K but improving baseline quality. "
     "Attack file changed to WADI_attackdataLABLE.csv with embedded per-row "
     "label column (Attack LABLE: -1=attack, 1=normal within attack window). "
     "A2 provides more precise ground truth: only 9,977 rows labeled as attacks "
     "vs A1's ~172K (entire attack file uniformly labeled)."),

    ("Columns dropped",
     "29 of 128 sensor columns removed before defining SENSOR_COLS: 4 columns "
     "were 100% NaN across the entire dataset (sensors never recorded), and 25 "
     "were constant-valued (no variation, zero discriminative power). Same "
     "columns dropped as A1 — consistent feature set across versions. All are "
     "documented in the run log and sensor_cols.json."),

    ("Time features",
     "Only observation_day and seconds_since_start are retained. Hour-of-day, "
     "day-of-week, and weekend flags were deliberately excluded — they have no "
     "physical relationship to sensor faults or cyber attacks in an ICS "
     "environment and would add spurious signal."),

    ("Leakage columns dropped",
     "Two columns dropped immediately after label assignment to prevent leakage: "
     "dataset_id (encodes source file — normal vs attack) and "
     "Attack LABLE (1:No Attack, -1:Attack) (directly encodes the label). "
     "Retaining either would be direct label leakage into the feature matrix."),

    ("Train/test split",
     "Sequence-level manual split. The 7 distinct attack sequences were identified "
     "by gaps > 1 minute between consecutive attack rows. Sequences were manually "
     "assigned to train (1, 5, 6, 7) or test (2, 3, 4) to ensure each split "
     "contains attacks of varying duration. Train: 4,336 attack rows (43.5%). "
     "Test: 5,641 attack rows (56.5%). Normal rows assigned by temporal proximity "
     "to neighboring attack sequences. Both splits contain normal and attack rows "
     "by design — required for supervised classification."),

    ("Why not standard WaDi protocol",
     "The standard protocol (normal → train, attack → test) was rejected because "
     "it leaves the model with no attack examples during training and no normal "
     "examples during evaluation — both undermine supervised classification. "
     "Sequence-level splitting preserves temporal integrity within each sequence "
     "while ensuring representative class coverage in both splits."),

    ("Known limitations",
     "Only 7 attack sequences total — sequence-level splitting produces coarse "
     "granularity. Sequences are highly unequal in size (57 to 5,134 rows), so "
     "sequence-count ratio does not equal row-count ratio. "
     "Normal rows within the attack window period (Oct 9–11) from the A2 attack "
     "file are labeled 0 and distributed across splits by temporal proximity."),

    ("Leakage risks",
     "Normalization stats must be fit on train split only — deferred to feature "
     "engineering notebook. Rolling window features must use only past "
     "observations and must be computed within each sequence to avoid bleeding "
     "across sequence boundaries — deferred to feature engineering notebook. "
     "Fault injection must not use information from attack labels — enforced "
     "in injection notebook."),

    ("Next step",
     "Fault injection notebook reads wadi_staged_*.parquet and injects synthetic "
     "sensor failures into normal rows in both train and test splits, producing a "
     "three-class dataset with label 2=fault. The curate/validate notebook then "
     "picks up from the injected parquet."),
]

print("Pipeline Reflection")
print("=" * 60)
for title, content in reflection:
    print(f"\n[{title}]")
    print(f"  {content}")

Pipeline Reflection

[Row definition]
  Each row represents one second of sensor readings from the WaDi A2 water distribution testbed. Label 0=normal operation, 1=cyber attack (derived from embedded Attack LABLE column in A2 attack file). Label 2=injected sensor fault will be added by the fault injection notebook.

[A2 vs A1 differences]
  Normal file changed to WADI_14days_new.csv — unstable periods removed, reducing rows from ~1.2M to ~784K but improving baseline quality. Attack file changed to WADI_attackdataLABLE.csv with embedded per-row label column (Attack LABLE: -1=attack, 1=normal within attack window). A2 provides more precise ground truth: only 9,977 rows labeled as attacks vs A1's ~172K (entire attack file uniformly labeled).

[Columns dropped]
  29 of 128 sensor columns removed before defining SENSOR_COLS: 4 columns were 100% NaN across the entire dataset (sensors never recorded), and 25 were constant-valued (no variation, zero discriminative power). Same columns dropped a

# This Continues with WaDi A2 Pipeline Notebook 2 - Fault Injection